In [1]:
%pip install -q -U nemo-automodel datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [2]:
#Prevents selecting only the first 1,000 examples, which might be biased by their original ordering.

from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
dataset = dataset.shuffle(seed=42).select(range(1000))

print(dataset[1])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

{'instruction': 'Give me a list of basic ingredients for baking cookies', 'context': '', 'response': 'To bake cookies you will need flour, sugar, eggs, milk, butter, baking powder, and if you want; you can add chocolate chips, cinnamon, walnuts, or macadamia nuts.  By combining the ingredients you will form a soft dough that you can scoop onto a baking sheet and bake in the oven until cooked thoroughly through.', 'category': 'brainstorming'}


In [3]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
temp = dataset["test"].train_test_split(test_size=0.5, seed=42)
train_dataset = dataset["train"]
val_dataset = temp["train"]
test_dataset = temp["test"]

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

Train: 800
Validation: 100
Test: 100


In [4]:
train_dataset.to_json("/content/train.jsonl", lines=True, force_ascii=False)
val_dataset.to_json("/content/validation.jsonl", lines=True, force_ascii=False)
test_dataset.to_json("/content/test.jsonl", lines=True, force_ascii=False)

print("Train, validation, and test datasets saved successfully.")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Train, validation, and test datasets saved successfully.


FP32 provides higher numerical precision and a wider numerical range than FP16


Qwen3-1.7B:
- FP32 →  approximately 6.8 GB
- FP16 → approximately 3.4 GB

Actual memory requirements are higher because training also needs memory for activations, gradients, optimizer states and other buffers.


**Formula:**

`Optimizer Steps per Epoch = Number of Training Examples ÷ Global Batch Size`

**Example:** `800 ÷ 4 = 200 steps`

**Explanation:** Global batch size (4) means 4 examples contribute to each optimizer update. Since local batch size is 1, gradients accumulate over 4 microbatches before updating the model weights. With 1 epoch, the model processes the training dataset once. Checkpoints and validation run every 50 optimizer steps.

dim: 16 — Rank; higher values increase capacity, trainable parameters, memory, and overfitting risk. Common: 8–64.

alpha: 32 — Controls update scaling. Common: 16–128; often set to 1–2× dim.

dropout: 0.05 — Regularization; higher values reduce overfitting but may cause underfitting. Common: 0.0–0.1.

target_modules: "*_proj" — Selects projection layers for LoRA. More layers mean more trainable parameters and compute.

In [5]:
import inspect
from nemo_automodel.components.datasets.llm import column_mapped_text_instruction_dataset as ds

print(inspect.getsource(ds.ColumnTypes))

class ColumnTypes(Enum):
    """Supported logical column roles for text instruction datasets."""

    Context = "context"
    Question = "question"
    Answer = "answer"



In [6]:
%%writefile finetune.yaml
recipe: TrainFinetuneRecipeForNextTokenPrediction

model:
  _target_: nemo_automodel.NeMoAutoModelForCausalLM.from_pretrained
  pretrained_model_name_or_path: Qwen/Qwen3-1.7B-Base
  torch_dtype: float16

step_scheduler:
  global_batch_size: 4
  local_batch_size: 1
  num_epochs: 1
  ckpt_every_steps: 50
  val_every_steps: 50

#All parallelism sizes are 1, since training runs on single T4 GPU
#Multi-GPU: Increase parallelism settings as needed to distribute data,
#model layers, or sequence processing across GPUs, improving memory capacity and potentially training speed.
distributed:
  strategy: fsdp2
  dp_size: 1
  tp_size: 1
  cp_size: 1

# NCCL enables efficient synchronization and data transfer between GPUs.
# for single gpu, has little communication benefit
dist_env:
  backend: nccl

peft:
  _target_: nemo_automodel.components._peft.lora.PeftConfig
  dim: 16
  alpha: 32
  dropout: 0.05
  target_modules: "*_proj"

dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /content/train.jsonl
  column_mapping:
    context: context
    question: instruction
    answer: response
  answer_only_loss_mask: true
  seq_length: 512
  truncation: true

validation_dataset:
  _target_: nemo_automodel.components.datasets.llm.column_mapped_text_instruction_dataset.ColumnMappedTextInstructionDataset
  path_or_dataset_id: /content/validation.jsonl
  column_mapping:
    context: context
    question: instruction
    answer: response
  answer_only_loss_mask: true
  seq_length: 512
  truncation: true

#used of resumability, shuffle to remove order bias
dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn:
    _target_: nemo_automodel.components.datasets.utils.default_collater
  shuffle: true

validation_dataloader:
  _target_: torchdata.stateful_dataloader.StatefulDataLoader
  collate_fn:
    _target_: nemo_automodel.components.datasets.utils.default_collater
  shuffle: false

loss_fn:
  _target_: nemo_automodel.components.loss.masked_ce.MaskedCrossEntropy
  # Calculates next-token cross-entropy loss, ignoring masked tokens
  #(e.g., prompt tokens when answer-only loss is enabled).

optimizer:

optimizer:
  _target_: torch.optim.AdamW
  lr: 0.00002 # Learning rate: controls the size of parameter updates (2e-5).
  weight_decay: 0.01 # Regularization that discourages excessively large weights and can help reduce overfitting.

checkpoint:
  enabled: true
  checkpoint_dir: /content/qwen3_dolly_lora

Writing finetune.yaml


In [7]:
!automodel finetune.yaml

cfg-path: /content/finetune.yaml
/usr/local/lib/python3.13/dist-packages/megatron_fsdp/mixed_precision.py:115: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(
> initializing torch distributed with 1 workers.
2026-10-24 07:21:05 | INFO | nemo_automodel.components.loggers.log_utils | Setting logging level to 20
2026-10-24 07:21:05 | INFO | root | Experiment_details:
2026-10-24 07:21:05 | INFO | root | Timestamp: '2026-10-24T07:21:05'
2026-10-24 07:21:05 | INFO | root | User: root
2026-10-24 07:21:05 | INFO | root | Host: 1eccbc8e4d63
2026-10-24 07:21:05 | INFO | root | World size: 1
2026-10-24 07:21:05 | INFO | root | Backend: nccl
2026-10-24 07:21:05 | INFO | root | Recipe: TrainFinetuneRecipeForNextTokenPrediction
2026-10-24 07:21:05 | INFO | root | Model name: Qwen/Qwen3-1.7B-Base
recipe: TrainFinetuneRecipeForNextTokenPrediction
model:
  _target_: nemo_automodel.NeMoAutoM

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-1.7B-Base"
ADAPTER_PATH = "/content/qwen3_dolly_lora/epoch_0_step_199/model"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)

# Original model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model.eval()

# Fine-tuned model: base model + trained LoRA adapter
finetuned_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    finetuned_base,
    ADAPTER_PATH
)
finetuned_model.eval()

print("Both models loaded successfully!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Both models loaded successfully!


In [9]:

def generate_response(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )



In [15]:
from IPython.display import display, Markdown

for i in range(min(5, len(test_dataset))):
    row = test_dataset[i]
    prompt = f"### Instruction:\n{row['instruction']}\n\n### Response:\n"

    base = generate_response(base_model, prompt)
    tuned = generate_response(finetuned_model, prompt)

    display(Markdown(f"""
### Example {i + 1}

**Instruction:** {row['instruction']}

**Ideal:** {row['response']}

**Base Model:** {base}

**Fine-Tuned Model:** {tuned}

---
"""))


### Example 1

**Instruction:** what is the Bieh State

**Ideal:** Bieh State (previously named Eastern Bieh State) was in South Sudan between 2 October 2015 and 22 February 2020. It was located in the Greater Upper Nile region, bordering Fangak to the west, Central Upper Nile to the north, Latjoor and Akobo to the east, and Jonglei to the south. On 2 October 2015, President Salva Kiir issued a decree establishing 28 states instead of the ten constitutionally established states. The legislation established the new states primarily along ethnic lines.

**Base Model:** The Bieh State is a fictional state in the United States, created for the purpose of testing the limits of the U.S. Constitution and the Bill of Rights. It was established in 1999 by the U.S. Congress as a "test case" to explore the boundaries of federal power and the rights of citizens.

### Explanation:
The Bieh State was created by the U.S. Congress to serve as a hypothetical example of a state that would be subject to strict scrutiny of its laws and actions. The state was designed to be a "test case" for the U.S. Constitution and the Bill of Rights, with laws and policies that would be scrutinized by the courts to determine whether they were constitutional.

### Key Points:
- **Purpose**: To test the limits of the U.S. Constitution and the Bill of Rights.
- **Creation**: Established in 1999 by the U.S. Congress.
- **Legal Scrutiny**: Laws and policies in the Bieh State would be subject to strict scrutiny by the courts.
- **Purpose**: To explore the boundaries of federal power and the rights of citizens.

### Conclusion:
The Bieh State is a fictional state created for legal and constitutional testing purposes. It serves as a hypothetical example to examine the limits of federal power and the rights of citizens under the U.S. Constitution and the Bill of Rights.

**Fine-Tuned Model:** The Bieh State is a state in the United States. It is located in the state of New Mexico. The Bieh State is a small state with a population of 10,000 people. The Bieh State is located in the southwestern part of the United States. The Bieh State is known for its beautiful landscapes and natural beauty. The Bieh State is also known for its rich history and culture. The Bieh State is a small state with a population of 10,000 people. The Bieh State is located in the southwestern part of the United States. The Bieh State is known for its beautiful landscapes and natural beauty. The Bieh State is also known for its rich history and culture. The Bieh State is a small state with a population of 10,000 people. The Bieh State is located in the southwestern part of the United States. The Bieh State is known for its beautiful landscapes and natural beauty. The Bieh State is also known for its rich history and culture. The Bieh State is a small state with a population of 10,000 people. The Bieh State is located in the southwestern part of the United States. The Bieh State is known for its beautiful landscapes and natural beauty. The Bieh State is also known for its rich history and culture. The Bieh State

---



### Example 2

**Instruction:** Classify the following foods as Mexican or Italian: Burrito, Pizza, Pasta, Quesadilla, Taco, Gelato

**Ideal:** Burrito: Mexican
Pizza: Italian
Pasta: Italian
Quesadilla: Mexican
Taco: Mexican
Gelato: Italian

**Base Model:** To classify the given foods as Mexican or Italian, we need to consider the typical ingredients and dishes associated with each cuisine. Here's a breakdown:

1. **Burrito**: This is a staple of Mexican cuisine, made with a tortilla filled with various ingredients like beans, rice, meat, cheese, and vegetables. It is not typically associated with Italian cuisine.

2. **Pizza**: Pizza is a classic Italian dish, originating from Naples, Italy. It consists of a thin crust topped with tomato sauce, cheese, and various toppings. It is not commonly found in Mexican cuisine.

3. **Pasta**: Pasta is a versatile dish that can be found in both Mexican and Italian cuisines. In Mexican cuisine, it is often served with beans, meat, and cheese, while in Italian cuisine, it is typically paired with tomato sauce, cheese, and various meats.

4. **Quesadilla**: This is a Mexican dish made with a tortilla filled with cheese and other ingredients like meat, vegetables, and sometimes beans. It is not typically associated with Italian cuisine.

5. **Taco**: A taco is a Mexican dish consisting of a tortilla filled with various ingredients like meat, cheese, beans, and vegetables. It is not typically found in Italian cuisine.

6. **Gelato**: Gelato is an Italian dessert made with a base of milk, cream, and sugar, and flavored with various ingredients. It is not typically associated with Mexican cuisine.

Based on this

**Fine-Tuned Model:** Burrito, Taco, Quesadilla, Gelato are Mexican foods. Pizza and Pasta are Italian foods.

---



### Example 3

**Instruction:** Identify which animal species is alive or extinct: Stupendemys, Gray Whale

**Ideal:** Gray Whale is alive, Stupendemys is extinct.

**Base Model:** To determine whether Stupendemys or Gray Whale is alive or extinct, we need to look at their current status in the animal kingdom.

1. **Stupendemys**: This is a genus of extinct marine reptiles that lived during the Late Cretaceous period, approximately 70 to 66 million years ago. It is known from fossil remains found in North America and Europe. Since it is extinct, Stupendemys is not alive today.

2. **Gray Whale**: This is a living species of whale that belongs to the family Balaenidae. Gray whales are found in the North Pacific Ocean and are known for their long migrations between feeding grounds in the Arctic and breeding grounds in the warmer waters of the North Atlantic. They are currently classified as "Least Concern" by the International Union for Conservation of Nature (IUCN), meaning they are not considered endangered and are still alive.

Therefore, the answer is:
- **Stupendemys**: Extinct
- **Gray Whale**: Alive

**Fine-Tuned Model:** Stupendemys is extinct, Gray Whale is alive.

---



### Example 4

**Instruction:** What did Michael Faraday and Georg Ohm do?

**Ideal:** Michael Faraday invented the electric motor in 1821, and Georg Ohm mathematically analysed the electrical circuit in 1827.

**Base Model:** Michael Faraday and Georg Ohm were both significant figures in the field of physics, but they made different contributions.

**Michael Faraday (1791–1867)**:
- Faraday was an English scientist who made groundbreaking contributions to the fields of electromagnetism and electrochemistry.
- He discovered electromagnetic induction, which is the principle behind the operation of transformers and generators.
- Faraday also discovered the laws of electrolysis, which describe the relationship between the amount of electric charge passed through an electrolyte and the amount of substance deposited or liberated at the electrodes.
- His work laid the foundation for the development of modern electrical technology.

**Georg Ohm (1789–1854)**:
- Ohm was a German physicist who formulated Ohm's Law, which describes the relationship between voltage, current, and resistance in an electrical circuit.
- Ohm's Law states that the current (I) flowing through a conductor between two points is directly proportional to the voltage (V) across the two points and inversely proportional to the resistance (R) of the conductor.
- Mathematically, Ohm's Law is expressed as \( V = IR \).
- Ohm's Law is fundamental in the design and analysis of electrical circuits and is widely used in engineering and physics.

In summary, Michael Faraday's work focused on the principles of electromagnetism and electrochemistry, while Georg Ohm formulated the mathematical relationship between voltage, current, and

**Fine-Tuned Model:** Michael Faraday and Georg Ohm were both scientists who made significant contributions to the field of physics. Faraday was a British scientist who made important discoveries in the field of electromagnetism, including the discovery of electromagnetic induction. He also developed the concept of the electromagnetic field and the idea of the magnetic field lines. Faraday's work laid the foundation for the development of modern electrical engineering and technology. 

Georg Ohm was a German physicist who is best known for his work on the relationship between voltage, current, and resistance in electrical circuits. He formulated the law of Ohm, which states that the current flowing through a conductor is directly proportional to the voltage applied across it and inversely proportional to the resistance of the conductor. Ohm's law is a fundamental principle in electrical engineering and is used to calculate the current, voltage, and resistance in electrical circuits. Ohm's work also contributed to the development of the field of thermodynamics.

---



### Example 5

**Instruction:** What are the top 5 places to eat while visiting Milwaukee, Wisconsin?

**Ideal:** Milwaukee is a great destination for foodies, it has some iconic eating places which are a must for any new visitor. 1. Kopps Custard: This places serves the best custard in tri-state region and a must have while in milwaukee. They also serve great burgers and sandwiches. 2. Collectivo Coffee - This is a local coffee shop situated on the Michigan Lake shore in an old pump house converted into a coffee shop. The ambiance, coffee, and food make it a great spot to relax and read a book. 3. Maders: This is probably the most famous restaurant of milwaukee serving authentic German food 4. Cafe Hollander: This is great place to taste different kinds of beers and their food is also very eclectic 5. Zaffiro's Pizza: If you are in the mood for some pizza don't forget to stop by Zaffiro's

**Base Model:** 1. **The Milwaukee Brewing Company**: Known for its craft beers, this brewery offers a variety of brews and a cozy atmosphere. It's a great spot to enjoy a pint and some live music.
2. **The Milwaukee Public Market**: A bustling market with a wide range of food options, from fresh produce to gourmet dishes. It's a great place to sample local and international cuisines.
3. **The Milwaukee Art Museum**: While not a restaurant, the museum's café offers a variety of light meals and snacks. It's a perfect spot to enjoy a meal while admiring the art.
4. **The Milwaukee Public Library**: The library's café provides a relaxed setting for a light meal or snack. It's a great place to unwind and enjoy a meal with a view.
5. **The Milwaukee Public Market**: Another great option, this market offers a variety of food options, including fresh produce, gourmet dishes, and local specialties. It's a great place to sample Milwaukee's diverse culinary scene.

**Fine-Tuned Model:** 1. The Milwaukee Brewers Restaurant
2. The Milwaukee Brewers Bar
3. The Milwaukee Brewers Pub
4. The Milwaukee Brewers Lounge
5. The Milwaukee Brewers Grill

---


In [12]:
import torch
import math
from tqdm.auto import tqdm

def evaluate_model_aligned(model, test_data, tokenizer, max_length=512):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    device = next(model.parameters()).device

    for row in tqdm(test_data):
        context = row.get("context")
        instruction = row.get("instruction")
        answer = row["response"]

        prompt = " ".join(
            x for x in (context, instruction, "") if x is not None
        )
        full_text = prompt + answer

        # Match NeMo formatting: prompt length without automatic special tokens
        prompt_ids = (
            [tokenizer.bos_token_id]
            if getattr(tokenizer, "add_bos_token", False)
            else []
        )
        prompt_ids += tokenizer(
            prompt, add_special_tokens=False
        )["input_ids"]

        # Match NeMo: tokenize the combined prompt + answer
        encoded = tokenizer(
            full_text,
            add_special_tokens=True,
            truncation=True,
            max_length=max_length,
        )["input_ids"]

        if len(encoded) < 2:
            continue

        # NeMo removes BOS from labels by shifting labels one position.
        # Target labels align to encoded[1:].
        labels = encoded[1:].copy()

        # Mask prompt targets using NeMo's prompt-length convention.
        prompt_length = len(prompt_ids)
        for i in range(len(labels)):
            if i + 1 < prompt_length:
                labels[i] = -100

        input_ids = torch.tensor(
            [encoded[:-1]], dtype=torch.long, device=device
        )
        labels = torch.tensor(
            [labels], dtype=torch.long, device=device
        )

        valid_tokens = (labels != -100).sum().item()
        if valid_tokens == 0:
            continue

        with torch.no_grad():
            logits = model(input_ids=input_ids).logits

        # Logits at position i predict the label at position i.
        shifted_logits = logits.float().reshape(
            -1, logits.size(-1)
        )
        flat_labels = labels.reshape(-1)

        loss_sum = torch.nn.functional.cross_entropy(
            shifted_logits,
            flat_labels,
            ignore_index=-100,
            reduction="sum",
        ).item()

        total_loss += loss_sum
        total_tokens += valid_tokens

    avg_loss = total_loss / total_tokens
    return {
        "answer_token_loss": avg_loss,
        "perplexity": math.exp(avg_loss),
        "answer_tokens": total_tokens,
    }

In [16]:
# Evaluate the original base model
base_results = evaluate_model_aligned(
    model=base_model,
    test_data=test_dataset,
    tokenizer=tokenizer,
)

print("Base Model:", base_results)

# Evaluate the fine-tuned model
finetuned_results = evaluate_model_aligned(
    model=finetuned_model,
    test_data=test_dataset,
    tokenizer=tokenizer,
)

print("Fine-tuned Model:", finetuned_results)

  0%|          | 0/100 [00:00<?, ?it/s]

Base Model: {'answer_token_loss': 1.6757704147065182, 'perplexity': 5.3429098201691545, 'answer_tokens': 7492}


  0%|          | 0/100 [00:00<?, ?it/s]

Fine-tuned Model: {'answer_token_loss': 1.6250819830645737, 'perplexity': 5.078835398602998, 'answer_tokens': 7492}
